# 🌿 Peatland Land-Cover Deep Learning Pipeline
**Open-data Sentinel-2 + CORINE · Random Forest vs U-Net · spatially honest evaluation · reproducible with a single upload step**

| | |
|---|---|
| **How to run** | `Runtime → Run all`. Block 3 shows an **optional Upload button** and waits at most 15 s; if you do not use it, the pipeline **continues automatically** with the clearly-labelled WorldCover proxy. For real CORINE results, drop the file into the Colab *Files* panel first - it is auto-detected and nothing waits. |
| **Best runtime** | GPU (T4) makes the U-Net ~5× faster; CPU also works (training is time-boxed). |
| **Modes** | `MODE = "demo"` (default, Block 2) is a fast preset for a live panel; set `MODE = "full"` for the complete run whose numbers you report. For a fast live run, execute the notebook once beforehand: the Sentinel-2 composite is cached for the session, so the second run skips the download. Real stage timings are printed at the end and saved in `reports/run_info.json`. |
| **Outputs** | `reports/` (metrics, data-quality report, 5 figures) + `models/unet_best.pth` + an auto-downloaded zip. |

**Pipeline:** 🛰️ composite → 🏷️ labels → 🧩 tiles → 🗺️ spatial split → 🔎 quality gate → 🌲 Random Forest → 🧠 U-Net → 📊 report

**Design principles:** open data only · no synthetic labels in the real pipeline · every number is computed in this run and saved to `reports/` · the pipeline *stops* rather than inventing data.

## Block 1 · Environment, helpers & reproducibility
**What it does:** installs only missing packages, imports everything, fixes all random seeds and detects CPU/GPU.

**Why it matters:** a fixed seed and a printed environment line make the run repeatable and auditable.

> 🎤 **Say to the panel:** *Every run is seeded and records its environment, so a colleague can reproduce my numbers.*

In [ ]:
import sys, subprocess, importlib, time, os, json, glob, zipfile, shutil, copy, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ---- install only what is missing (fast on Colab: most packages are preinstalled) ----
_REQ = {"pystac_client": "pystac-client", "planetary_computer": "planetary-computer", "odc.stac": "odc-stac",
        "segmentation_models_pytorch": "segmentation-models-pytorch", "rasterio": "rasterio",
        "sklearn": "scikit-learn", "scipy": "scipy", "pyproj": "pyproj", "matplotlib": "matplotlib", "tqdm": "tqdm"}
def _missing():
    out = []
    for mod, pkg in _REQ.items():
        try: importlib.import_module(mod)
        except Exception: out.append(pkg)
    return out
_m = _missing()
if _m:
    print("Installing:", ", ".join(_m))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_m])

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import rasterio
from rasterio.warp import reproject, Resampling
from pyproj import Transformer
from affine import Affine
import pystac_client, planetary_computer, odc.stac
from odc.geo.geobox import GeoBox
from dask.diagnostics import ProgressBar
from scipy import ndimage as ndi
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, cohen_kappa_score
import torch, torch.nn as nn
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm
from IPython.display import display, HTML

# ---- reproducibility ----
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(os.cpu_count() or 2)
Path("reports").mkdir(exist_ok=True); Path("models").mkdir(exist_ok=True); Path("data/raw").mkdir(parents=True, exist_ok=True)

# ---- presentation helpers (pure HTML, no extra dependencies) ----
def banner(step, title, sub=""):
    display(HTML(f"""<div style="margin:28px 0 12px;padding:14px 20px;border-radius:10px;color:#fff;
      background:linear-gradient(90deg,#0f2027,#203a43,#2c5364);box-shadow:0 3px 8px rgba(0,0,0,.25);font-family:Segoe UI,Arial,sans-serif">
      <span style="background:#4ade80;color:#052e16;border-radius:50%;padding:3px 10px;font-weight:700;margin-right:10px">{step}</span>
      <span style="font-size:20px;font-weight:700">{title}</span>
      <div style="opacity:.8;margin-top:4px;font-size:13px">{sub}</div></div>"""))

def _note(msg, bg, fg, bar, icon):
    display(HTML(f'<div style="padding:9px 14px;margin:6px 0;background:{bg};color:{fg};border-left:5px solid {bar};'
                 f'border-radius:4px;font-family:Segoe UI,Arial,sans-serif;font-size:14px">{icon} {msg}</div>'))
def ok(m):   _note(m, "#dcfce7", "#14532d", "#22c55e", "✅")
def info(m): _note(m, "#e0f2fe", "#0c4a6e", "#0ea5e9", "ℹ️")
def warn(m): _note(m, "#fef3c7", "#78350f", "#f59e0b", "⚠️")

def html_table(rows, header, highlight=None, width="100%"):
    th = "".join(f'<th style="padding:6px 10px;text-align:left;background:#1e293b;color:#fff">{h}</th>' for h in header)
    body = ""
    for i, r in enumerate(rows):
        bg = "#fef9c3" if highlight is not None and i == highlight else ("#f8fafc" if i % 2 else "#ffffff")
        body += "<tr>" + "".join(f'<td style="padding:5px 10px;background:{bg};color:#0f172a">{c}</td>' for c in r) + "</tr>"
    display(HTML(f'<table style="border-collapse:collapse;width:{width};font-family:Segoe UI,Arial,sans-serif;font-size:13px;'
                 f'box-shadow:0 1px 4px rgba(0,0,0,.15)"><tr>{th}</tr>{body}</table>'))

TIMINGS = {}
class stage:
    """Context manager that records wall-time per pipeline stage."""
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a): TIMINGS[self.name] = time.time() - self.t

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU) - Runtime > Change runtime type > T4 GPU makes the U-Net ~5x faster"
display(HTML(f"""<div style="padding:16px;border-radius:12px;background:linear-gradient(135deg,#ecfeff,#dcfce7);font-family:Segoe UI,Arial,sans-serif">
<h2 style="margin:0;color:#064e3b">🌿 peatland-landcover-dl · Sentinel-2 + CORINE · RF vs U-Net</h2>
<div style="margin-top:8px;color:#134e4a;font-size:14px">Pipeline: 🛰️ composite → 🏷️ labels → 🧩 tiles → 🗺️ spatial split → 🔎 QC → 🌲 RF → 🧠 U-Net → 📊 report</div>
<div style="margin-top:6px;color:#475569;font-size:13px">Python {sys.version.split()[0]} · torch {torch.__version__} · CPU cores {os.cpu_count()} · GPU: {gpu} · seed {SEED}</div></div>"""))

## Block 2 · Configuration
**What it does:** one `CFG` dictionary holds dates, bands, tile/block sizes, model settings and the CORINE → 7-class lookup tables.

**Why it matters:** no magic numbers hidden in code; changing a setting needs one edit. The class scheme matches the study design (peat bogs = class 4).

> 🎤 **Say to the panel:** *All choices are in one place; the class mapping is explicit and handles both CORINE raster encodings.*

In [ ]:
# ===================== ALL SETTINGS LIVE HERE (single source of truth) =====================
CFG = dict(
    year=2018, months=("05", "09"), max_cloud=30, max_scenes_per_tile=6,   # imagery selection
    bands=["B02", "B03", "B04", "B08", "B11", "B12"], res=10, epsg=32629,   # UTM 29N covers all of Ireland's west/midlands
    area_fraction=1.0,          # 1.0 = full study box; 0.5 = central half (about 4x faster, handy for a quick rehearsal)
    min_peat_px=50_000,         # a study box must hold >= 5 km2 of peat labels, else the next box is tried
    tile=128, block=256,        # 128 px tiles nested exactly inside 256 px (2.56 km) spatial blocks
    buffer_tiles=1,             # one-tile (1.28 km) exclusion ring between TEST and train/validation data (strict, reported split)
    val_buffer_tiles=0,         # ring between validation and train; 0 keeps ~45% of the area for training instead of ~27% (validation only drives early stopping)
    min_valid_tile=0.30, split=(0.70, 0.15, 0.15),
    rf_max_px=150_000, rf_trees=60,
    unet_batch=16, unet_lr=2e-3, unet_epochs=12, unet_patience=3, unet_min_epochs=5, unet_tiles_per_epoch=640,
    unet_budget_s=600 if torch.cuda.is_available() else 1500,  # safety cap only: the reported run should finish its fixed epochs (flagged in metrics.json if not)
    seed=SEED)

MODE = "demo"   # "demo" = faster preset for a live panel (fewer scenes, smaller Random Forest) | "full" = complete run - use THIS for any numbers you report
if MODE == "demo":
    CFG.update(max_scenes_per_tile=4, rf_max_px=80_000, rf_trees=40,     # keeps the full study area; fewer scenes, smaller Random Forest, shorter U-Net time cap
               unet_budget_s=180 if torch.cuda.is_available() else 420)
else:                                                                     # "full": the reported run trains longer (the time cap above still applies)
    CFG.update(unet_epochs=30, unet_patience=5)

AREAS = [dict(name="Irish Midlands bogs",     bbox=(-7.90, 53.10, -7.50, 53.40)),
         dict(name="Connemara blanket bog",   bbox=(-9.90, 53.40, -9.50, 53.60))]   # automatic fallback

CLASS_NAMES = ["Artificial", "Agriculture", "Forest", "Open semi-natural", "Peat bogs", "Other wetlands", "Water"]
CLASS_COLORS = ["#d62728", "#ffd966", "#2e7d32", "#a8d08d", "#6d4c41", "#26a69a", "#1e88e5", "#ffffff"]  # last = nodata
CMAP = ListedColormap(CLASS_COLORS)
PEAT = 4

# CORINE: the Copernicus GeoTIFF stores indices 1-44 (legend order) OR 3-digit codes depending on the product; both handled.
CLC_CODES = [111,112,121,122,123,124,131,132,133,141,142,211,212,213,221,222,223,231,241,242,243,244,311,312,313,
             321,322,323,324,331,332,333,334,335,411,412,421,422,423,511,512,521,522,523]
def _code_to_class(c):
    if 111 <= c <= 142: return 0
    if 211 <= c <= 244: return 1
    if c in (311, 312, 313): return 2
    if c in (321, 322, 324, 331, 332, 333, 334, 335): return 3
    if c == 412: return 4
    if c in (411, 421, 422, 423): return 5
    if 511 <= c <= 523: return 6
    return 255
CODE_LUT = np.full(1024, 255, np.uint8)
for c in CLC_CODES: CODE_LUT[c] = _code_to_class(c)
IDX_LUT = np.full(256, 255, np.uint8)
for i, c in enumerate(CLC_CODES, start=1): IDX_LUT[i] = CODE_LUT[c]

# ESA WorldCover -> same 7 classes. NOTE: WorldCover has NO peat class; "herbaceous wetland (90)" is only a PROXY.
WC_LUT = np.full(256, 255, np.uint8)
for k, v in {50: 0, 40: 1, 10: 2, 20: 3, 30: 3, 60: 3, 100: 3, 90: 4, 95: 5, 80: 6}.items(): WC_LUT[k] = v

banner("2", "Configuration", "one dictionary, no magic numbers hidden in the code")
html_table([["MODE", MODE]] + [[k, str(v)] for k, v in CFG.items() if k not in ("bands",)] + [["bands", ", ".join(CFG["bands"]) + " + NDVI + NDMI"]],
           ["setting", "value"], width="70%")

## Block 3 · Upload widget · CORINE label file
**What it does:** auto-detects a CORINE file already in the session; otherwise shows an optional upload button with a 15 s countdown and then **continues automatically**. Unpacks nested Copernicus zips and picks the GeoTIFF.

**Why it matters:** CORINE needs a free Copernicus login so it cannot be downloaded automatically. If you cancel, a clearly-labelled WorldCover **proxy** is used and flagged in every report.

> 🎤 **Say to the panel:** *I never silently substitute data: without CORINE the notebook says so and labels the results as proxy.*

In [ ]:
def _resolve_tif(p, depth=0):
    """Return a GeoTIFF path from a .tif or from a (possibly nested) Copernicus .zip."""
    p = Path(p)
    if p.suffix.lower() in (".tif", ".tiff"): return p
    if p.suffix.lower() == ".zip" and depth < 4:
        out = Path("data/raw") / p.stem; out.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(p) as z: z.extractall(out)
        tifs = [t for t in out.rglob("*") if t.suffix.lower() in (".tif", ".tiff") and "ovr" not in t.name.lower()]
        if tifs:
            tifs.sort(key=lambda t: ("clc" not in t.name.lower(), -t.stat().st_size)); return tifs[0]
        for z2 in out.rglob("*.zip"):
            r = _resolve_tif(z2, depth + 1)
            if r: return r
    return None

def _already_there():
    pats = ["*CLC*", "*clc*", "*corine*", "*CORINE*", "U20*"]
    for d in (Path("."), Path("/content"), Path("data/raw")):
        for pat in pats:
            for f in d.glob(pat):
                if f.suffix.lower() in (".tif", ".tiff", ".zip"): return f
    return None

UPLOAD_WAIT_S = 15   # how long the pipeline waits for a file before continuing automatically (0 = do not show the button)

def timed_upload(wait_s):
    """Optional upload button with a countdown. Never blocks longer than wait_s seconds; any problem -> returns None and the run continues."""
    if wait_s <= 0: return None
    try:
        import base64
        from google.colab import output
    except ImportError:
        warn("Not running in Colab: place the CORINE file next to the notebook and re-run this cell."); return None
    try:
        display(HTML("""
        <div style="font-family:Segoe UI,Arial,sans-serif;padding:12px 16px;border:2px dashed #0ea5e9;border-radius:10px;background:#f0f9ff;max-width:660px">
          <b>📂 Optional: choose the CORINE file (.tif or .zip)</b>
          <div style="margin:8px 0"><input type="file" id="corine_in" accept=".tif,.tiff,.zip"></div>
          <div id="corine_msg" style="color:#0c4a6e;font-size:13px">Waiting <span id="corine_t">__T__</span> s - then the pipeline continues automatically with the labelled WorldCover proxy.</div>
        </div>
        <script>
        window._corine = {state: "idle", name: "", data: null};
        document.getElementById("corine_in").addEventListener("change", function(e) {
          const f = e.target.files[0]; if (!f) return;
          window._corine.state = "reading"; window._corine.name = f.name;
          document.getElementById("corine_msg").innerText = "Reading " + f.name + " ...";
          const r = new FileReader();
          r.onload = function() { window._corine.data = r.result.split(",")[1]; window._corine.state = "done";
            document.getElementById("corine_msg").innerText = "Received " + f.name + " - continuing."; };
          r.onerror = function() { window._corine.state = "error"; };
          r.readAsDataURL(f);
        });
        let left = __T__;
        const iv = setInterval(function() {
          if (window._corine.state !== "idle") { clearInterval(iv); return; }
          left -= 1;
          if (left <= 0) { clearInterval(iv); document.getElementById("corine_msg").innerText = "No file chosen - continuing with the WorldCover proxy."; }
          else document.getElementById("corine_t").innerText = left;
        }, 1000);
        </script>""".replace("__T__", str(int(wait_s)))))
        t0, state = time.time(), "idle"
        while time.time() - t0 < wait_s and state == "idle":
            time.sleep(0.5); state = output.eval_js("window._corine.state")
        if state == "reading":                                   # a file was picked in time: let the browser finish reading it
            t1 = time.time()
            while state == "reading" and time.time() - t1 < 300:
                time.sleep(0.5); state = output.eval_js("window._corine.state")
        if state != "done": return None
        name = Path(output.eval_js("window._corine.name")).name
        n = int(output.eval_js("window._corine.data.length")); CH = 2_097_152     # multiple of 4 -> valid base64 chunks
        out = Path("data/raw") / name
        with open(out, "wb") as fh:
            for i in range(0, n, CH): fh.write(base64.b64decode(output.eval_js(f"window._corine.data.slice({i},{i + CH})")))
        return out
    except Exception as e:
        warn(f"Upload button unavailable ({type(e).__name__}) - continuing automatically."); return None

banner("3", "Upload the CORINE label file", "optional upload button · the run continues automatically after the countdown with the clearly-labelled WorldCover proxy")
label_file, label_path = _already_there(), None
if label_file:
    info(f"Found <b>{label_file}</b> already in the session - skipping the upload dialog (fully automatic).")
else:
    info("Download <b>CORINE Land Cover 2018 (raster, 100 m)</b> from "
         "<a href='https://land.copernicus.eu/en/products/corine-land-cover' target='_blank'>land.copernicus.eu</a> "
         "(free login) - the .zip or .tif is fine. Tip: the Copernicus download tool lets you clip to Ireland, which makes the file small.")
    label_file = timed_upload(UPLOAD_WAIT_S)       # optional - never blocks the run

if label_file:
    label_path = _resolve_tif(label_file)
    if label_path is None: warn("Could not find a GeoTIFF inside that file - falling back to the proxy labels.")
LABEL_MODE = "corine" if label_path else "worldcover_proxy"
if LABEL_MODE == "corine": ok(f"CORINE ready: <code>{label_path}</code> ({label_path.stat().st_size/1e6:.1f} MB)")
else: warn("No CORINE file - using <b>ESA WorldCover as a PROXY</b>. It has no peat class (herbaceous wetland is substituted), "
           "so peat-bog results are NOT comparable to CORINE results and are labelled as such in every report.")

## Block 4 · Study area & labels
**What it does:** builds one 10 m UTM grid (trimmed to whole spatial blocks), warps CORINE onto it, auto-detects index-vs-code encoding and checks peat coverage - trying the Connemara box if the Midlands box has too little peat.

**Why it matters:** imagery and labels share the exact same grid, so misalignment (a classic silent bug) cannot happen.

> 🎤 **Say to the panel:** *I verify the study box really contains peat-bog labels before spending time on imagery.*

In [ ]:
banner("4", "Study area & labels", "grid is built once, labels are warped onto it - imagery and labels can never drift apart")
catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)

def make_geobox(bbox, frac=1.0, res=10, epsg=32629, block=256):
    """UTM grid snapped to the 10 m Sentinel-2 lattice, trimmed to a whole number of spatial blocks (no padding needed)."""
    tr = Transformer.from_crs(4326, epsg, always_xy=True)
    pts = [tr.transform(x, y) for x in (bbox[0], bbox[2]) for y in (bbox[1], bbox[3])]
    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
    cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
    hw, hh = (max(xs) - min(xs)) / 2 * frac, (max(ys) - min(ys)) / 2 * frac
    xmin, ymin = np.floor((cx - hw) / res) * res, np.floor((cy - hh) / res) * res
    W = int((2 * hw) / res) // block * block
    H = int((2 * hh) / res) // block * block
    return GeoBox((H, W), Affine(res, 0, xmin, 0, -res, ymin + H * res), f"EPSG:{epsg}")

def gb_lonlat(gb):
    tr = Transformer.from_crs(gb.crs.to_wkt(), 4326, always_xy=True)
    H, W = tuple(gb.shape); a = gb.affine
    pts = [tr.transform(*(a * (c, r))) for c in (0, W) for r in (0, H)]
    return (min(p[0] for p in pts), min(p[1] for p in pts), max(p[0] for p in pts), max(p[1] for p in pts))

def load_corine(path, gb):
    dst = np.zeros(tuple(gb.shape), np.int16)
    with rasterio.open(path) as src:
        reproject(source=rasterio.band(src, 1), destination=dst, dst_transform=gb.affine, dst_crs=gb.crs.to_wkt(),
                  resampling=Resampling.nearest, src_nodata=src.nodata, dst_nodata=0)
    vals = np.unique(dst[dst > 0])
    n_code, n_idx = int(np.isin(vals, CLC_CODES).sum()), int(np.isin(vals, np.arange(1, 45)).sum())
    if n_idx > n_code: scheme, lab = "index(1-44)", IDX_LUT[np.clip(dst, 0, 255)]
    else:              scheme, lab = "code(3-digit)", CODE_LUT[np.clip(dst, 0, 1023)]
    return lab.astype(np.uint8), scheme

def load_worldcover(gb):
    items = list(catalog.search(collections=["esa-worldcover"], bbox=gb_lonlat(gb)).item_collection())
    if not items: raise RuntimeError("No WorldCover items found for this area.")
    def yr(it):
        s = str(it.properties.get("start_datetime") or it.datetime or "0000")
        return int(s[:4]) if s[:4].isdigit() else 0
    best = min({yr(i) for i in items}, key=lambda y: abs(y - CFG["year"]))
    ds = odc.stac.load([i for i in items if yr(i) == best], bands=["map"], geobox=gb, resampling="nearest")
    arr = np.asarray(ds["map"].values); arr = arr[0] if arr.ndim == 3 else arr
    return WC_LUT[arr.astype(np.uint8)], f"worldcover_{best}"

rows, chosen, best_try = [], None, None
with stage("labels"):
    for area in AREAS:
        gb_try = make_geobox(area["bbox"], CFG["area_fraction"], CFG["res"], CFG["epsg"], CFG["block"])
        lab_try, scheme = load_corine(label_path, gb_try) if LABEL_MODE == "corine" else load_worldcover(gb_try)
        n_peat = int((lab_try == PEAT).sum()); H_, W_ = tuple(gb_try.shape)
        rows.append([area["name"], f"{W_*CFG['res']/1000:.1f} x {H_*CFG['res']/1000:.1f} km", scheme, f"{n_peat:,} px ({n_peat*100/1e6:.1f} km2)"])
        cand = dict(area=area, gb=gb_try, lab=lab_try, scheme=scheme, n_peat=n_peat)
        if best_try is None or n_peat > best_try["n_peat"]: best_try = cand
        if n_peat >= CFG["min_peat_px"]: chosen = cand; break
html_table(rows, ["area tried", "size", "label scheme", "peat-bog label pixels"])
if chosen is None:
    if best_try["n_peat"] == 0:
        raise RuntimeError("No peat-bog pixels in any study box - check the label file / bounding boxes. Stopping instead of inventing data.")
    chosen = best_try; warn("Peat coverage is below the target; continuing with the best box - peat metrics will be noisy.")
area, gb, lab_raw, scheme = chosen["area"], chosen["gb"], chosen["lab"], chosen["scheme"]
label_source = "corine" if LABEL_MODE == "corine" else scheme
H, W = tuple(gb.shape)
ok(f"Study area: <b>{area['name']}</b> · grid {H} x {W} px ({H*W/1e6:.1f} M pixels) · CRS {gb.crs} · labels: <b>{label_source}</b> ({scheme})")

## Block 5 · Sentinel-2 median composite
**What it does:** searches Planetary Computer, keeps the clearest scenes per MGRS tile, masks cloud/shadow with the SCL band **per scene**, then takes the median over time - lazily, in chunks.

**Why it matters:** masking before the median removes clouds properly; chunking keeps memory low; limiting scenes cuts download time.

> 🎤 **Say to the panel:** *Cloud masking happens per scene before compositing, not afterwards.*

In [ ]:
banner("5", "Sentinel-2 median composite", "cloud-masked per scene, then median over time · lazy + chunked so it fits in Colab RAM")
import calendar
def pick_scenes(bbox):
    y = int(CFG["year"]); m0, m1 = int(CFG["months"][0]), int(CFG["months"][1])
    last_day = calendar.monthrange(y, m1)[1]                      # real last day of the end month (28-31), not a hard-coded 30
    found = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime=f"{y}-{m0:02d}-01/{y}-{m1:02d}-{last_day:02d}",
                           query={"eo:cloud_cover": {"lt": CFG["max_cloud"]}}).item_collection()
    by = {}
    for it in found: by.setdefault(it.properties.get("s2:mgrs_tile", "?"), []).append(it)
    sel = []
    for t, its in by.items():                      # best (least cloudy) scenes per MGRS tile -> full coverage, minimal download
        its.sort(key=lambda i: i.properties.get("eo:cloud_cover", 100)); sel += its[:CFG["max_scenes_per_tile"]]
    return sel, {t: len(v) for t, v in by.items()}, len(found)

import hashlib
def _enc(a):   # float32 DN -> uint16 (65535 = no data); rounding to 0.5 DN is far below sensor noise
    return np.where(np.isfinite(a), np.clip(np.rint(a), 0, 65534), 65535).astype(np.uint16)
def _dec(u):
    a = u.astype("float32"); a[u == 65535] = np.nan; return a
_key = hashlib.md5(json.dumps([area["bbox"], list(map(int, gb.shape)), list(gb.affine)[:6],
                               {k: CFG[k] for k in ("year", "months", "max_cloud", "max_scenes_per_tile", "bands", "epsg")}], default=str).encode()).hexdigest()[:10]
CACHE = Path("data/interim") / f"composite_{_key}.npz"; CACHE.parent.mkdir(parents=True, exist_ok=True)
_cached = CACHE.exists()      # same area + dates + settings built earlier in this Colab session -> reuse (delete data/interim to force a rebuild)

with stage("sentinel2"):
    if _cached:
        _z = np.load(CACHE); comp = _dec(_z["comp"]); _m = json.loads(str(_z["meta"]))
        N_SCENES, BASELINES, OFFSET = _m["n_scenes"], _m["baselines"], _m["offset"]
        info(f"Composite <b>loaded from cache</b> (built earlier in this session from {N_SCENES} scenes, processing baselines {BASELINES}). Delete <code>data/interim</code> to rebuild it from Planetary Computer.")
    else:
        info(f"No cached composite for this configuration (<code>{CACHE.name}</code>). Cached files present: {[p.name for p in CACHE.parent.glob('*.npz')] or 'none'} - building it from Planetary Computer now (this is the slow step).")
        sel, per_tile, n_found = pick_scenes(gb_lonlat(gb))
        if not sel: raise RuntimeError("No Sentinel-2 scenes found - widen the dates or cloud limit in CFG.")
        # Processing baseline >= 04.00 adds a +1000 DN offset to L2A reflectance; it must be removed or NDVI/NDMI are biased.
        BASELINES = sorted({str(i.properties.get("s2:processing_baseline", "?")) for i in sel})
        def _ge4(b):
            try: return float(b) >= 4.0
            except ValueError: return None
        _flags = {_ge4(b) for b in BASELINES}
        OFFSET = 1000.0 if _flags == {True} else 0.0
        info(f"Sentinel-2 processing baselines in use: {BASELINES} -> reflectance offset removed: {OFFSET:.0f} DN")
        if len(_flags) > 1: warn("Scenes mix processing baselines (some with the +1000 offset, some without). A per-scene correction is NOT applied here, so indices may be slightly biased - note this in the limitations.")
        info(f"{n_found} scenes under {CFG['max_cloud']}% cloud across tiles {per_tile} → using the {len(sel)} clearest.")
        ds = odc.stac.load(sel, bands=CFG["bands"] + ["SCL"], geobox=gb, groupby="solar_day", chunks={"x": 1024, "y": 1024},
                           resampling={**{b: "bilinear" for b in CFG["bands"]}, "SCL": "nearest"}, fail_on_error=False)
        refl = ds[CFG["bands"]].to_array("band").astype("float32")
        good = ds["SCL"].isin([4, 5, 6])            # vegetation, bare soil, water -> drops cloud, shadow, snow, defective and unclassified (haze)
        masked = refl.where(good & (refl > 0)) - OFFSET
        with ProgressBar():
            comp = masked.chunk({"time": -1, "band": 1}).median("time", skipna=True).compute().values   # (6, H, W) in DN
        del ds, refl, good, masked
        N_SCENES = len(sel)
        _u = _enc(comp); comp = _dec(_u)      # identical rounding with or without the cache
        np.savez(CACHE, comp=_u, meta=json.dumps(dict(n_scenes=N_SCENES, baselines=BASELINES, offset=OFFSET)))
cover = float(np.isfinite(comp).all(0).mean())
ok(f"Composite {'loaded from cache' if _cached else 'built'} in {TIMINGS['sentinel2']:.0f} s · {N_SCENES} scenes · valid coverage {cover*100:.1f}%")

## Block 6 · Feature engineering & label cleaning
**What it does:** scales reflectance, adds NDVI and NDMI (8 channels), marks pixels without imagery as nodata (255) and plots imagery next to labels.

**Why it matters:** NDMI (moisture) is informative for wet peat; nodata is ignored by every model and metric.

> 🎤 **Say to the panel:** *Eight channels: six bands plus two indices; nodata is excluded everywhere.*

In [ ]:
banner("6", "Feature engineering & label cleaning", "reflectance scaling, NDVI/NDMI, nodata = 255")
with stage("features"):
    refl = np.clip(comp / 10000.0, 0, 1.5).astype("float32")
    B = {n: refl[i] for i, n in enumerate(CFG["bands"])}
    ndvi = (B["B08"] - B["B04"]) / (B["B08"] + B["B04"] + 1e-6)     # greenness
    ndmi = (B["B08"] - B["B11"]) / (B["B08"] + B["B11"] + 1e-6)     # canopy / surface moisture (key for wet peat)
    X = np.concatenate([refl, ndvi[None], ndmi[None]]).astype("float32")
    FEATURES = CFG["bands"] + ["NDVI", "NDMI"]
    finite = np.isfinite(X).all(0)
    lab = lab_raw.copy(); lab[~finite] = 255           # no imagery -> no label (ignored everywhere)

def stretch(a):
    lo, hi = np.nanpercentile(a, (2, 98)); return np.clip((a - lo) / (hi - lo + 1e-9), 0, 1)
rgb = np.stack([stretch(B["B04"]), stretch(B["B03"]), stretch(B["B02"])], -1)
s = 3
fig, ax = plt.subplots(1, 2, figsize=(15, 6.2))
ax[0].imshow(np.nan_to_num(rgb[::s, ::s]), interpolation="nearest"); ax[0].set_title("Sentinel-2 composite (true colour)", fontweight="bold")
ax[1].imshow(np.where(lab == 255, 7, lab)[::s, ::s], cmap=CMAP, vmin=0, vmax=7, interpolation="nearest")
ax[1].set_title(f"7-class labels ({label_source})", fontweight="bold")
for a in ax: a.axis("off")
ax[1].legend(handles=[mpatches.Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i]) for i in range(7)], loc="lower left", fontsize=8, framealpha=.9)
plt.tight_layout(); plt.savefig("reports/01_study_area.png", dpi=130); plt.show()

cnt = np.bincount(lab[lab != 255], minlength=7)
html_table([[f"{i}", CLASS_NAMES[i], f"{cnt[i]:,}", f"{cnt[i]/max(cnt.sum(),1)*100:.2f}%"] for i in range(7)],
           ["id", "class", "pixels", "share"], highlight=PEAT, width="60%")

## Block 7 · Tiling & spatial block split
**What it does:** cuts 128×128 tiles nested exactly inside 2.56 km blocks, assigns **whole blocks** to train/val/test (70/15/15), adds a one-tile exclusion ring around the **test** blocks (validation, used only for early stopping, is not ringed from training by default - `val_buffer_tiles`), and standardises features with training statistics only.

**Why it matters:** random pixel splits leak information through spatial autocorrelation and inflate scores; block splits with a buffer give a realistic estimate of performance on new areas. The seed search only checks that peat exists in all splits and never looks at model scores.

> 🎤 **Say to the panel:** *I split by spatial block with a buffer to avoid leakage, and normalise with training statistics only.*

In [ ]:
banner("7", "Tiling & spatial block split", "whole 2.56 km blocks go to train / val / test · exclusion ring around the test blocks · leakage reduced by whole-block assignment and an exclusion ring")
T, BK = CFG["tile"], CFG["block"]; TPB = BK // T
nTr, nTc = H // T, W // T
with stage("split"):
    valid_frac = (lab != 255).reshape(nTr, T, nTc, T).mean((1, 3))
    tile_ok = valid_frac >= CFG["min_valid_tile"]
    block_ok = tile_ok.reshape(H // BK, TPB, W // BK, TPB).any((1, 3))
    peat_tile = (lab == PEAT).reshape(nTr, T, nTc, T).sum((1, 3))
    ub = np.argwhere(block_ok)

    def assign(seed):
        rng = np.random.default_rng(seed); perm = rng.permutation(len(ub))
        n_te = max(1, round(CFG["split"][2] * len(ub))); n_va = max(1, round(CFG["split"][1] * len(ub)))
        bs = np.full(block_ok.shape, -1, np.int8)
        for k, i in enumerate(perm): bs[tuple(ub[i])] = 2 if k < n_te else (1 if k < n_te + n_va else 0)
        ts = np.kron(bs, np.ones((TPB, TPB), np.int8)); ts[~tile_ok] = -1       # tile grid: -1 unused 0 train 1 val 2 test
        st = np.ones((3, 3), bool)
        if CFG["buffer_tiles"] > 0:                                              # (iterations=0 would mean "until stable", so guard it)
            near_test = ndi.binary_dilation(ts == 2, st, iterations=CFG["buffer_tiles"])
            ts[near_test & ((ts == 0) | (ts == 1))] = 3                          # 3 = exclusion buffer around TEST
        if CFG["val_buffer_tiles"] > 0:
            near_val = ndi.binary_dilation(ts == 1, st, iterations=CFG["val_buffer_tiles"])
            ts[near_val & (ts == 0)] = 3
        return bs, ts

    def peat_balance(ts):
        kept = peat_tile[(ts >= 0) & (ts < 3)].sum()
        if kept == 0: return False, (np.nan,) * 3
        sh = tuple(peat_tile[ts == k].sum() / kept for k in (0, 1, 2))
        return (sh[0] >= .50 and sh[1] >= .07 and sh[2] >= .07), sh

    chosen_seed, bsplit, TS, shares = None, None, None, None
    for sd in range(CFG["seed"], CFG["seed"] + 300):       # deterministic search for a split that contains peat in every part
        bs, ts = assign(sd); good_split, sh = peat_balance(ts)
        if good_split: chosen_seed, bsplit, TS, shares = sd, bs, ts, sh; break
    if TS is None: chosen_seed = CFG["seed"]; bsplit, TS = assign(chosen_seed); shares = peat_balance(TS)[1]; warn("No perfectly peat-balanced split found; using the default seed.")

    SPLITPIX = np.kron(TS, np.ones((T, T), np.int8))          # pixel-level split map
    tr_pix = (SPLITPIX == 0) & (lab != 255)
    mu = X[:, tr_pix].mean(1); sd_ = X[:, tr_pix].std(1) + 1e-6   # statistics from TRAINING pixels only
    Xn = ((X - mu[:, None, None]) / sd_[:, None, None]).astype("float32"); Xn[~np.isfinite(Xn)] = 0

names = {0: "Train", 1: "Validation", 2: "Test", 3: "Buffer (excluded)"}
rows = [[names[k], f"{(TS==k).sum():,}", f"{(TS==k).sum()*T*T/1e6:.2f} M px", (f"{shares[k]*100:.1f}%" if k < 3 and not np.isnan(shares[k]) else "-")] for k in range(4)]
html_table(rows, ["split", "tiles (128 x 128)", "area", "share of peat pixels kept"])
ok(f"Block-split seed {chosen_seed} (chosen so train, val and test all contain peat - decided from labels only, never from model scores)")

fig, ax = plt.subplots(figsize=(8.5, 7))
cm = ListedColormap(["#ffffff", "#1f77b4", "#2ca02c", "#d62728", "#9ca3af"])
ax.imshow(TS + 1, cmap=cm, vmin=0, vmax=4, interpolation="nearest"); ax.set_title(f"Spatial split (each cell = 1.28 km tile) · seed {chosen_seed}", fontweight="bold")
ax.legend(handles=[mpatches.Patch(color=c, label=l) for c, l in zip(["#1f77b4", "#2ca02c", "#d62728", "#9ca3af"], ["Train", "Validation", "Test", "Buffer"])], loc="lower right")
ax.axis("off"); plt.tight_layout(); plt.savefig("reports/02_spatial_split.png", dpi=130); plt.show()
del comp, refl, B, ndvi, ndmi, rgb

## Block 8 · Data-quality gate
**What it does:** recomputes independent checks from the final arrays (CRS, resolution, shapes, NaNs, block purity, buffer distance, class presence, nodata) and writes `reports/data_quality.json`.

**Why it matters:** critical failures stop the run; softer issues (e.g. a class absent from a split) are surfaced as warnings instead of being hidden.

> 🎤 **Say to the panel:** *Quality control is automated and fails loudly; the report is committed with the results.*

In [ ]:
banner("8", "Data-quality gate", "independent checks recomputed from the final arrays · pipeline stops if a critical check fails")
checks = []
def chk(name, passed, detail, critical=True):
    checks.append(dict(name=name, status="pass" if passed else ("fail" if critical else "warn"), detail=detail))

res_x, res_y = abs(gb.affine.a), abs(gb.affine.e)
chk("CRS is projected (UTM 29N) and shared by imagery & labels", str(gb.crs).upper().endswith(str(CFG["epsg"])), str(gb.crs))
chk("Pixel size is 10 m", abs(res_x - CFG["res"]) < 1e-6 and abs(res_y - CFG["res"]) < 1e-6, f"{res_x} x {res_y} m")
chk("Image and label shapes match", X.shape[1:] == lab.shape == tuple(gb.shape), f"{X.shape} vs {lab.shape}")
chk("No NaN/Inf in features at labelled pixels", bool(np.isfinite(X[:, lab != 255]).all()), "checked on every labelled pixel")
chk("Labels only contain classes 0-6 or 255", bool(np.isin(np.unique(lab), list(range(7)) + [255]).all()), str(np.unique(lab).tolist()))

# block purity: every spatial block belongs to at most one of train / val / test
v = TS.reshape(H // BK, TPB, W // BK, TPB).transpose(0, 2, 1, 3).reshape(H // BK, W // BK, -1)
m = (v >= 0) & (v <= 2)
mn, mx = np.where(m, v, 9).min(-1), np.where(m, v, -9).max(-1)
chk("No spatial block appears in two splits", bool(((mn == mx) | (~m.any(-1))).all()), f"{int(m.any(-1).sum())} blocks checked")
def _gap(a, b):   # smallest chessboard distance (tiles) from any tile of split b to the nearest tile of split a
    return int(ndi.distance_transform_cdt(TS != a, metric="chessboard")[TS == b].min()) if (TS == a).any() and (TS == b).any() else None
for _a, _b, _txt, _need in ((2, 0, "Train tiles are >= buffer distance from every test tile", CFG["buffer_tiles"] + 1),
                            (2, 1, "Validation tiles are >= buffer distance from every test tile", CFG["buffer_tiles"] + 1),
                            (1, 0, "Train tiles are >= buffer distance from every validation tile", CFG["val_buffer_tiles"] + 1)):
    _g = _gap(_a, _b)
    if _g is not None: chk(_txt, _g >= _need, f"minimum gap = {_g} tiles (required {_need})")
chk("Train, validation and test are all non-empty", all((TS == k).any() for k in (0, 1, 2)), f"{[(int((TS==k).sum())) for k in (0,1,2)]} tiles")

split_counts = {}
for k, nm in ((0, "train"), (1, "val"), (2, "test")):
    sel_ = (SPLITPIX == k) & (lab != 255)
    split_counts[nm] = np.bincount(lab[sel_], minlength=7).tolist()
for nm in ("train", "val", "test"):
    chk(f"Peat bogs present in {nm}", split_counts[nm][PEAT] > 0, f"{split_counts[nm][PEAT]:,} px", critical=False)
    missing = [CLASS_NAMES[i] for i in range(7) if split_counts[nm][i] == 0]
    chk(f"All 7 classes present in {nm}", not missing, "missing: " + (", ".join(missing) if missing else "none"), critical=False)
nod = 1 - valid_frac[(TS >= 0) & (TS < 3)]
chk("Nodata fraction in kept tiles is below 70%", float(nod.max()) <= 0.70, f"mean {nod.mean()*100:.1f}% · max {nod.max()*100:.1f}%")

icon = {"pass": "✅", "warn": "⚠️", "fail": "❌"}
html_table([[icon[c["status"]], c["name"], c["detail"]] for c in checks], ["", "check", "result"])
status = "fail" if any(c["status"] == "fail" for c in checks) else ("warn" if any(c["status"] == "warn" for c in checks) else "pass")
json.dump(dict(status=status, label_source=label_source, label_scheme=scheme, area=area["name"], seed=chosen_seed,
               grid=[H, W], checks=checks, class_counts_per_split={k: dict(zip(CLASS_NAMES, v)) for k, v in split_counts.items()}),
          open("reports/data_quality.json", "w"), indent=2)
if status == "fail": raise AssertionError("Data-quality gate FAILED - see table above and reports/data_quality.json")
ok(f"Quality gate: <b>{status.upper()}</b> · report saved to reports/data_quality.json")

## Block 9 · Random Forest baseline
**What it does:** trains on square-root-balanced pixel samples - the same effective class prior as the U-Net's square-root class weights, so the two models are compared fairly and predicts all unseen test pixels in batches.

**Why it matters:** a strong, interpretable baseline - the U-Net has to beat it to justify its cost.

> 🎤 **Say to the panel:** *The baseline uses only spectral information, so any U-Net gain reflects spatial context.*

In [ ]:
banner("9", "Random Forest baseline", "per-pixel spectral model · square-root-balanced sampling (same effective class prior as the U-Net)")
TEST_MASK = (SPLITPIX == 2) & (lab != 255)
with stage("random_forest"):
    rng = np.random.default_rng(CFG["seed"])
    flat_lab, flat_tr = lab.ravel(), tr_pix.ravel()
    picks = []
    n_max = int(np.bincount(flat_lab[flat_tr], minlength=7).max())          # largest training class
    for c in range(7):
        idx = np.flatnonzero(flat_tr & (flat_lab == c))
        if len(idx):                                                         # per-class sample ~ sqrt(class size)
            k = max(1, min(len(idx), int(CFG["rf_max_px"] * np.sqrt(len(idx) / n_max))))
            picks.append(rng.choice(idx, k, replace=False))
    picks = np.concatenate(picks)
    Xf = Xn.reshape(len(FEATURES), -1)
    rf = RandomForestClassifier(n_estimators=CFG["rf_trees"], max_depth=18, min_samples_leaf=3, class_weight=None,
                                n_jobs=-1, random_state=CFG["seed"]).fit(Xf[:, picks].T, flat_lab[picks])
    te_idx = np.flatnonzero(TEST_MASK.ravel())
    pred_rf = np.empty(len(te_idx), np.uint8)
    for i in range(0, len(te_idx), 400_000):                   # batched prediction keeps RAM flat
        pred_rf[i:i + 400_000] = rf.predict(Xf[:, te_idx[i:i + 400_000]].T)
y_true = flat_lab[te_idx]
ok(f"Random Forest trained on {len(picks):,} sampled pixels and applied to {len(te_idx):,} unseen test pixels in {TIMINGS['random_forest']:.0f} s")
imp = sorted(zip(FEATURES, rf.feature_importances_), key=lambda t: -t[1])
info("Feature importance: " + " · ".join(f"<b>{n}</b> {v*100:.0f}%" for n, v in imp[:5]))

## Block 10 · U-Net (ResNet-18 encoder)
**What it does:** trains a slim U-Net from scratch with weighted cross-entropy (nodata ignored), flip/rotation augmentation, early stopping on validation loss and a hard time budget; predictions are painted back onto the map.

**Why it matters:** class weights counter the rarity of peat; the time budget keeps the demo predictable; the best checkpoint by validation loss is restored.

> 🎤 **Say to the panel:** *Early stopping and checkpointing are driven by validation blocks, never by the test blocks.*

In [ ]:
banner("10", "U-Net (ResNet-18 encoder)", "trained from scratch · weighted cross-entropy ignoring nodata · flips + 90° rotations · time-boxed with early stopping")
def gather(code):
    idx = np.argwhere(TS == code)
    if len(idx) == 0: return None, None, idx
    xs = np.stack([Xn[:, r*T:(r+1)*T, c*T:(c+1)*T] for r, c in idx])
    ys = np.stack([lab[r*T:(r+1)*T, c*T:(c+1)*T] for r, c in idx]).astype(np.int64)
    return torch.from_numpy(xs), torch.from_numpy(ys), idx

with stage("unet"):
    Xtr, Ytr, _ = gather(0); Xva, Yva, _ = gather(1)
    cnt = np.bincount(Ytr[Ytr != 255].numpy().ravel(), minlength=7).astype(np.float64)
    w = np.where(cnt > 0, (cnt.sum() / (7 * np.maximum(cnt, 1))) ** 0.5, 0.0); w = w / w[cnt > 0].mean()
    crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE), ignore_index=255)

    model = smp.Unet("resnet18", encoder_weights=None, in_channels=len(FEATURES), classes=7,
                     decoder_channels=(128, 64, 32, 16, 16)).to(DEVICE)           # slim decoder = fast on CPU
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["unet_lr"], weight_decay=1e-4)
    use_amp = DEVICE.type == "cuda"
    try: scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    arng = np.random.default_rng(CFG["seed"])
    BS, hist, best, bad, best_state, t0 = CFG["unet_batch"], [], float("inf"), 0, None, time.time()

    def augment(xb, yb):
        k = int(arng.integers(4)); xb, yb = torch.rot90(xb, k, (2, 3)), torch.rot90(yb, k, (1, 2))
        if arng.random() < .5: xb, yb = torch.flip(xb, (3,)), torch.flip(yb, (2,))
        return xb, yb

    rows_live = []; budget_hit = False
    TPE = CFG["unet_tiles_per_epoch"]
    def epoch_order():   # exactly TPE tiles per epoch: whole shuffled passes over the training tiles (old randperm(...)[:TPE] returned only len(Xtr) tiles)
        return torch.from_numpy(np.concatenate([arng.permutation(len(Xtr)) for _ in range(-(-TPE // len(Xtr)))])[:TPE])
    print(f"{len(Xtr)} training tiles · {TPE} tiles per epoch · {-(-TPE // BS)} optimiser steps per epoch")
    for ep in range(1, CFG["unet_epochs"] + 1):
        model.train(); perm = epoch_order(); tl, nb = 0.0, 0
        for i in range(0, len(perm), BS):
            ib = perm[i:i + BS]; xb, yb = augment(Xtr[ib], Ytr[ib]); xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp): loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tl += float(loss); nb += 1
            if time.time() - t0 > CFG["unet_budget_s"]: budget_hit = True; break
        model.eval(); vl, vn = 0.0, 0
        with torch.no_grad():
            for i in range(0, len(Xva), 32):
                xb, yb = Xva[i:i + 32].to(DEVICE), Yva[i:i + 32].to(DEVICE)
                with torch.autocast(device_type=DEVICE.type, enabled=use_amp): vl += float(crit(model(xb), yb)) * len(xb)
                vn += len(xb)
        tl, vl = tl / max(nb, 1), vl / max(vn, 1); hist.append((tl, vl))
        flag = ""
        if vl < best - 1e-4: best, bad, best_state, flag = vl, 0, copy.deepcopy(model.state_dict()), " ⭐ best"
        else: bad += 1
        print(f"epoch {ep:2d} | train {tl:.3f} | val {vl:.3f} | {time.time()-t0:5.0f}s{flag}")
        if bad >= CFG["unet_patience"] and ep >= CFG["unet_min_epochs"]: print("early stopping (validation loss stopped improving)"); break
        if time.time() - t0 > CFG["unet_budget_s"]: budget_hit = True; print("time budget reached - this is NOT a fixed-epoch run"); break
    if best_state is None: best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state); torch.save(best_state, "models/unet_best.pth")

    # inference on the held-out test tiles, painted back onto the full grid
    Xte, _, te_tiles = gather(2); pred_map = np.full((H, W), 255, np.uint8); model.eval()
    with torch.no_grad():
        for i in range(0, len(Xte), 32):
            out = model(Xte[i:i + 32].to(DEVICE)).argmax(1).cpu().numpy().astype(np.uint8)
            for (r, c), p in zip(te_tiles[i:i + 32], out): pred_map[r*T:(r+1)*T, c*T:(c+1)*T] = p
    pred_unet = pred_map.ravel()[te_idx]
ok(f"U-Net trained for {len(hist)} epochs in {TIMINGS['unet']:.0f} s · best validation loss {best:.3f} · checkpoint saved to models/unet_best.pth")
if len(hist) < CFG["unet_min_epochs"]: warn(f"U-Net ran only {len(hist)} epochs (minimum {CFG['unet_min_epochs']}) - it is under-trained; BatchNorm statistics have not settled. Use a GPU runtime or raise unet_budget_s.")
fig, ax = plt.subplots(figsize=(6, 3.4)); ax.plot([h[0] for h in hist], "o-", label="train"); ax.plot([h[1] for h in hist], "s-", label="validation")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("weighted CE loss (log scale)"); ax.set_title("U-Net training curve", fontweight="bold"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("reports/03_unet_training.png", dpi=130); plt.show()

## Block 11 · Evaluation on unseen blocks
**What it does:** scores both models on the **same** test pixels: accuracy, macro F1, kappa, per-class precision/recall/F1, row-normalised confusion matrices, test maps and a peat hit/miss/false-alarm map.

**Why it matters:** peat is reported separately because it is the class of interest; macro F1 averages only classes present in the test set.

> 🎤 **Say to the panel:** *Peat recall and precision tell different stories - the map shows where peat is missed or falsely predicted.*

In [ ]:
banner("11", "Evaluation on unseen spatial blocks", "identical test pixels for both models · peat bogs reported separately")
LABS = list(range(7))
def score(y, p):
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=LABS, zero_division=0)
    present = sup > 0
    return dict(accuracy=float(accuracy_score(y, p)), macro_f1=float(f1[present].mean()), kappa=float(cohen_kappa_score(y, p)),
                peat_f1=float(f1[PEAT]) if sup[PEAT] else None, peat_precision=float(pr[PEAT]) if sup[PEAT] else None,
                peat_recall=float(rc[PEAT]) if sup[PEAT] else None, peat_support_px=int(sup[PEAT]),
                per_class={CLASS_NAMES[i]: dict(precision=float(pr[i]), recall=float(rc[i]), f1=float(f1[i]), support=int(sup[i])) for i in LABS})
res = {"RandomForest": score(y_true, pred_rf), "UNet": score(y_true, pred_unet)}
if res["UNet"]["kappa"] < 0.10:
    warn(f"<b>The U-Net did not converge</b> (kappa {res['UNet']['kappa']:.2f}) - it was probably stopped too early or hit the time cap. Do NOT interpret its scores; re-run, on a GPU runtime if possible.")
meta = dict(label_source=label_source, label_scheme=scheme, area=area["name"], test_pixels=int(len(y_true)), seed=CFG["seed"], split_seed=int(chosen_seed),
            unet_epochs_run=len(hist), unet_time_budget_hit=bool(budget_hit), mode=MODE, unet_converged=bool(res["UNet"]["kappa"] >= 0.10),
            note=("Labels are CORINE (100 m, 25 ha minimum mapping unit)." if label_source == "corine"
                  else "PROXY labels: ESA WorldCover has no peat class (herbaceous wetland substituted). Not comparable to CORINE results."))
for k, v in res.items(): json.dump({**v, "model": k, **meta}, open(f"reports/metrics_{k}.json", "w"), indent=2)
json.dump({**meta, "models": res, "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()}}, open("reports/metrics.json", "w"), indent=2)

def cm_norm(y, p):
    m = confusion_matrix(y, p, labels=LABS).astype(float); return m / np.maximum(m.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(1, 3, figsize=(20, 5.6), gridspec_kw=dict(width_ratios=[1, 1, 1.05]))
for a, (nm, p) in zip(ax[:2], [("Random Forest", pred_rf), ("U-Net", pred_unet)]):
    m = cm_norm(y_true, p); a.imshow(m, cmap="Blues", vmin=0, vmax=1)
    for i in LABS:
        for j in LABS: a.text(j, i, f"{m[i,j]:.2f}", ha="center", va="center", fontsize=8, color="white" if m[i, j] > .5 else "black")
    a.set_xticks(LABS); a.set_yticks(LABS); a.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8); a.set_yticklabels(CLASS_NAMES, fontsize=8)
    a.set_xlabel("predicted"); a.set_ylabel("reference"); a.set_title(f"{nm} · row-normalised confusion", fontweight="bold")
wd = .38; xs = np.arange(7)
ax[2].bar(xs - wd/2, [res["RandomForest"]["per_class"][n]["f1"] for n in CLASS_NAMES], wd, label="Random Forest", color="#2a9d8f")
ax[2].bar(xs + wd/2, [res["UNet"]["per_class"][n]["f1"] for n in CLASS_NAMES], wd, label="U-Net", color="#e76f51")
ax[2].set_xticks(xs); ax[2].set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8); ax[2].set_ylim(0, 1.18); ax[2].set_ylabel("F1")
ax[2].axvspan(PEAT - .5, PEAT + .5, color="#6d4c41", alpha=.12); ax[2].set_title("Per-class F1 (peat highlighted)", fontweight="bold"); ax[2].legend(); ax[2].grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig("reports/04_results.png", dpi=130); plt.show()

f = lambda v: "n/a" if v is None else f"{v:.3f}"
best_name = max(res, key=lambda k: (res[k]["peat_f1"] or -1))
html_table([[k, f"{v['accuracy']:.3f}", f"{v['macro_f1']:.3f}", f"{v['kappa']:.3f}", f(v["peat_f1"]), f(v["peat_precision"]), f(v["peat_recall"])] for k, v in res.items()],
           ["model", "overall accuracy", "macro F1", "kappa", "🟤 peat F1", "peat precision", "peat recall"],
           highlight=list(res).index(best_name), width="85%")

# ---- maps on the test blocks (reference | RF | U-Net | peat agreement for the better model) ----
pred_rf_map = np.full(H * W, 255, np.uint8); pred_rf_map[te_idx] = pred_rf; pred_rf_map = pred_rf_map.reshape(H, W)
ref_map = np.where(TEST_MASK, lab, 255); s = 3
def show(a, m, title):
    a.imshow(np.where(m == 255, 7, m)[::s, ::s], cmap=CMAP, vmin=0, vmax=7, interpolation="nearest"); a.set_title(title, fontweight="bold", fontsize=11); a.axis("off")
fig, ax = plt.subplots(1, 4, figsize=(22, 5.8))
show(ax[0], ref_map, "Reference (test blocks only)")
show(ax[1], np.where(TEST_MASK, pred_rf_map, 255), "Random Forest")
show(ax[2], np.where(TEST_MASK, pred_map, 255), "U-Net")
bm = pred_rf_map if best_name == "RandomForest" else pred_map
agree = np.zeros((H, W), np.uint8); agree[TEST_MASK] = 1
agree[TEST_MASK & (lab == PEAT) & (bm == PEAT)] = 2; agree[TEST_MASK & (lab == PEAT) & (bm != PEAT)] = 3; agree[TEST_MASK & (lab != PEAT) & (bm == PEAT)] = 4
ax[3].imshow(agree[::s, ::s], cmap=ListedColormap(["#ffffff", "#e5e7eb", "#16a34a", "#dc2626", "#f59e0b"]), vmin=0, vmax=4, interpolation="nearest")
ax[3].set_title(f"Peat bogs · {best_name}", fontweight="bold", fontsize=11); ax[3].axis("off")
ax[3].legend(handles=[mpatches.Patch(color=c, label=l) for c, l in zip(["#16a34a", "#dc2626", "#f59e0b"], ["correct peat", "missed peat", "false alarm"])], loc="lower left", fontsize=8)
ax[0].legend(handles=[mpatches.Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i]) for i in range(7)], loc="lower left", fontsize=7, framealpha=.9)
plt.tight_layout(); plt.savefig("reports/05_test_maps.png", dpi=130); plt.show()

## Block 12 · Run summary & export
**What it does:** prints stage timings, a computed headline, honest limitations, zips `reports/` and triggers the download.

**Why it matters:** the reports are generated by the code, so the README numbers can match `metrics.json` exactly.

> 🎤 **Say to the panel:** *Everything I show is generated by this run, with limitations stated alongside the results.*

In [ ]:
banner("12", "Run summary & export", "everything below is computed from this run - nothing is typed in by hand")
total = sum(TIMINGS.values())
html_table([[k.replace("_", " "), f"{v:.0f} s", f"{v/total*100:.0f}%"] for k, v in TIMINGS.items()] + [["<b>total (excl. upload wait)</b>", f"<b>{total:.0f} s ({total/60:.1f} min)</b>", ""]],
           ["stage", "time", "share"], width="45%")
if label_source != "corine": warn("These results use PROXY labels (WorldCover). State this clearly if you show them - peat F1 is not a real peat-bog score.")
if MODE == "demo": warn("<b>DEMO MODE</b> - fewer scenes, smaller Random Forest and a shorter U-Net time cap for a fast live run (same study area). Do not quote these numbers as final results; set MODE = \"full\" in Block 2 for the reported run.")
rf_, un_ = res["RandomForest"], res["UNet"]
def _verdict(rf, un, margin=0.02):
    """Compare on macro F1, peat F1 and kappa (not one metric alone); gaps below `margin` count as ties."""
    u_ahead, r_ahead = [], []
    for k, lbl in (("macro_f1", "macro F1"), ("peat_f1", "peat F1"), ("kappa", "kappa")):
        x, y = rf.get(k), un.get(k)
        if x is None or y is None: continue
        if y - x > margin: u_ahead.append(lbl)
        elif x - y > margin: r_ahead.append(lbl)
    if u_ahead and r_ahead: return f"mixed result: U-Net ahead on {', '.join(u_ahead)}; Random Forest ahead on {', '.join(r_ahead)}"
    if u_ahead: return f"U-Net outperforms the Random Forest on {', '.join(u_ahead)}"
    if r_ahead: return f"Random Forest outperforms the U-Net on {', '.join(r_ahead)}"
    return "no clear difference between the two models"
verdict = "the U-Net did NOT converge, so this comparison is invalid" if un_["kappa"] < 0.10 else _verdict(rf_, un_)
info(f"<b>Headline:</b> macro F1 RF {rf_['macro_f1']:.3f} vs U-Net {un_['macro_f1']:.3f}; peat F1 RF {f(rf_['peat_f1'])} vs U-Net {f(un_['peat_f1'])}; "
     f"kappa RF {rf_['kappa']:.3f} vs U-Net {un_['kappa']:.3f} → {verdict} on this split (test peat pixels: {rf_['peat_support_px']:,}). "
     "One split, no confidence interval: gaps under ~0.02 are noise.")
LABEL_NOTE = ("CORINE is 100 m with a 25 ha minimum mapping unit (coarse against 10 m imagery); check its reference year against the 2018 imagery" if label_source == "corine"
              else f"labels are a PROXY ({scheme}: 10 m WorldCover, no peat class, and its release year differs from the 2018 imagery)")
info("<b>Read these as honest limitations:</b> " + LABEL_NOTE + "; summer 2018 was unusually hot and dry in Ireland, so the composite may not be typical; "
     "one area, one season, one block split with few peat blocks (results are noisy); no field validation; small U-Net trained from scratch. "
     "This is not a peatland-condition assessment.")
json.dump(dict(timings_s={k: round(v, 1) for k, v in TIMINGS.items()}, python=sys.version.split()[0], torch=torch.__version__, device=str(DEVICE),
               n_scenes=N_SCENES, composite_cached=bool(_cached), s2_processing_baselines=BASELINES, s2_offset_removed_dn=OFFSET,
               unet_epochs_run=len(hist), unet_time_budget_hit=bool(budget_hit), unet_train_tiles=int(len(Xtr)), unet_tiles_per_epoch=int(CFG["unet_tiles_per_epoch"]), mode=MODE), open("reports/run_info.json", "w"), indent=2)
shutil.make_archive("peatland_reports", "zip", "reports")
ok("Saved: reports/metrics.json · data_quality.json · run_info.json · 5 figures · models/unet_best.pth · peatland_reports.zip")
try:
    from google.colab import files as _f; _f.download("peatland_reports.zip")
except Exception: pass